In [1]:
#Importing required libraries
import pandas as pd
import yfinance as yf
from tqdm import tqdm
from datetime import date, datetime, timedelta
import numpy as np
import warnings

# Code where you want to ignore warnings
warnings.filterwarnings("ignore")

In [2]:
# Function to retrieve data from Yahoo Finance API
def retrieve_api(symbol):
    return yf.Ticker(symbol)

# *********************************************************************************************************************
# Firmographics Section
# *********************************************************************************************************************

# Function to extract company data from API result
def get_company_data(result):
    # Creating a function to handle missing values from API while assigning variables
    def assign_variable(result, value):
        try:
            variable = result.info[value]
        except Exception as e:
            variable = "Not Found"  # Assigning a default value in case of an error
        return variable
    
    # Allotting key parameters from the API result
    industry = assign_variable(result, 'industry')
    dividendRate = assign_variable(result, 'dividendRate')
    sector = assign_variable(result, 'sector')
    website = assign_variable(result, 'website')
    beta = assign_variable(result, 'beta')
    name = assign_variable(result, 'longName')
    earningsGrowth = assign_variable(result, 'earningsGrowth')
    revenueGrowth = assign_variable(result, 'revenueGrowth')
    firstTradeDate = assign_variable(result, 'firstTradeDateEpochUtc')
    sharesOutstanding = assign_variable(result, 'sharesOutstanding')
    trailingEps = assign_variable(result, 'trailingEps')
    summary = assign_variable(result, 'longBusinessSummary')
    symbol = assign_variable(result, 'symbol')
    previousClose = assign_variable(result, 'previousClose')
    previousVolume = assign_variable(result, 'volume')
    marketCap = assign_variable(result, 'marketCap')
    fiftyTwoWeekLow = assign_variable(result, 'fiftyTwoWeekLow')
    fiftyTwoWeekHigh = assign_variable(result, 'fiftyTwoWeekHigh')
    pegRatio = assign_variable(result, 'pegRatio')
    
    # Determine the exchange based on the symbol suffix
    if symbol[-3:] == ".NS":
        exchange = "NSE"
    elif symbol[-3:] == ".BO":
        exchange = "BSE"
    
    # Compiling the data into a DataFrame
    company_parameters = ["Name", "Symbol", "Exchange", "Website", "Description", "Sector", "Industry", "Beta",
                          "Dividend Rate", "Earnings Growth", "Revenue Growth", "Shares OutStanding",
                          "EPS (trailing)", "Market Cap"]
    company_values = [name, symbol, exchange, website, summary, sector, industry, beta, dividendRate, earningsGrowth,
                      revenueGrowth, sharesOutstanding, trailingEps, marketCap]
    params_df = pd.DataFrame({'Parameter': company_parameters, "Value": company_values})
     
    return params_df.set_index('Parameter').transpose()

# Function to compile firmographics for a single company
def compile_single_firmo(symbol):
    # Retrieve raw data from API
    result = retrieve_api(symbol)
    # Process raw data to get company data
    company_data = get_company_data(result)
    # Make tweaks to the data, such as replacing sector values
    replace_map = {'Consumer Cyclical': 'FMCG', 'Consumer Defensive': 'FMCG'}
    company_data['Sector'] = company_data['Sector'].replace(replace_map)
    # Return the compiled company data
    return company_data

# Function to compile firmographics for all companies in a list
def compile_all(list_of_symbols): 
    df = pd.DataFrame()
    # Loop through the list of symbols, execute compile_single for each, and append the results to the DataFrame
    for i in tqdm(list_of_symbols):
        try:
            df_s = compile_single_firmo(i)
            if df_s.empty:
                continue
            else:
                df = pd.concat([df, df_s], axis=0)
        except:
            continue
            
    # Final cleaning: Remove companies with 'Not Found' sector or industry
    df = df[df['Sector'] != 'Not Found']
    df = df[df['Industry'] != 'Not Found']
    # Add sector and industry median PEs (not implemented in this code snippet)
    # df = add_sector_industry_pe_median(df)
    
    return df


In [ ]:
# Read the company list from a CSV file
in_companies = pd.read_csv('company_list.csv')

# Compile firmographics for all companies in the list
compiled = compile_all(in_companies.SYMBOL)
# Add creation timestamp to the DataFrame
compiled["created_on"] = datetime.now()

# Save the compiled firmographics to a CSV file
compiled.to_csv('db_firmo.csv', index=False)

In [ ]:

# *********************************************************************************************************************
#Stock Data
# *********************************************************************************************************************

# Function to retrieve historical data from Yahoo Finance API for a given result
def get_hist_data(result):
    # Get today's date
    date_today = date.today() + timedelta(days=1)
    # Fetch historical data from 2020-01-01 until today for daily intervals
    historical = result.history(period="2y")

    # Remove days with zero trading volume (representing market holidays)
    historical = historical[historical['Volume'] != 0]
    
    # Sort the DataFrame by date in descending order and add a 'Trading Day' column representing the trading day number,
    # i.e., if today is 5 July (day 1), then 4 July will be 2, and so on.
    historical = historical.sort_values('Date', ascending=False).reset_index().reset_index()
    historical = historical.rename(columns={'index': 'Trading Day'})
    historical['Trading Day'] = historical['Trading Day'] + 1
    
    # Create a new column with date only (removing time information)
    date_only = []
    for i in historical['Date']:
        x = str(pd.to_datetime(i).year).zfill(2) + "-" + str(pd.to_datetime(i).month).zfill(2) + "-" + str(pd.to_datetime(i).day).zfill(2)
        date_only.append(np.datetime64(x))
    historical['date_only'] = date_only
    
    # Keep only values from the last 2 years
    historical_sample = historical[historical['date_only'] > historical['date_only'][0] - np.timedelta64(2, "Y")]
    
    return historical_sample

# Function to calculate Exponential Moving Average (EMA) for a given DataFrame, series, and interval
def calc_ema(df, interval):
    df = df.sort_values('Trading Day', ascending=False)
    ema = df['Close'].ewm(span=interval, adjust=False).mean()
    col_name = "ema_" + str(interval)
    df[col_name] = ema
    df = df.sort_values('Trading Day', ascending=True)
    return df

# Function to calc macd
def calc_macd(df):
    df = df.sort_values('Trading Day', ascending=False)

    df['ema_12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['ema_26'] = df['Close'].ewm(span=25, adjust=False).mean()
    df['macd'] = df['ema_12'] - df['ema_26']
    df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['macde_histogram'] = df['macd'] - df['macd_signal']
    df.drop(['ema_12','ema_26'],axis=1, inplace= True)
    
    df = df.sort_values('Trading Day', ascending=True)
    
    return df

# Function to calculate Simple Moving Average (SMA) for a given DataFrame and interval
def calc_sma(df, interval):
    df = df.sort_values('Trading Day', ascending=False)
    sma = df['Close'].rolling(interval).mean()
    col_name = "sma_" + str(interval)
    df[col_name] = sma
    df = df.sort_values('Trading Day', ascending=True)
    return df

# Function to calculate Relative Strength Index (RSI) for a given DataFrame and window_length
def calc_rsi(df, window_length):
    # Calculate differences between closing prices of consecutive days
    diff = []
    for i in range(df.shape[0] - 1):
        diff.append(df.Close[i] - df.Close[i + 1])
    diff.append(np.nan)
    df['diff'] = diff
    df = df.sort_values('Trading Day', ascending=False)

    # Calculate gains and losses based on differences
    df['gain'] = df['diff'].clip(lower=0).round(2)
    df['loss'] = df['diff'].clip(upper=0).abs().round(2)

    # Calculate average gains and losses for the given window length
    df['avg_gain'] = df['gain'].rolling(window=window_length, min_periods=window_length).mean()[:window_length + 1]
    df['avg_loss'] = df['loss'].rolling(window=window_length, min_periods=window_length).mean()[:window_length + 1]

    # Calculate average gains and losses for the rest of the DataFrame
    for i, row in enumerate(df['avg_gain'].iloc[window_length + 1:]):
        df['avg_gain'].iloc[i + window_length + 1] = \
            (df['avg_gain'].iloc[i + window_length] * (window_length - 1) + df['gain'].iloc[i + window_length + 1]) / window_length

    for i, row in enumerate(df['avg_loss'].iloc[window_length + 1:]):
        df['avg_loss'].iloc[i + window_length + 1] = \
            (df['avg_loss'].iloc[i + window_length] * (window_length - 1) + df['loss'].iloc[i + window_length + 1]) / window_length

    # Calculate Relative Strength (RS) and RSI
    df['rs'] = df['avg_gain'] / df['avg_loss']
    df['rsi'] = 100 - (100 / (1.0 + df['rs']))
    df = df.drop(['gain', 'loss', 'rs', 'avg_gain', 'avg_loss', 'diff'], axis=1)
    df = df.sort_values('Trading Day', ascending=True)
    return df

# Function to calculate Average Directional Index (ADX) for a given DataFrame and period
def calc_ADX(data: pd.DataFrame, period: int):
    """
    Computes the ADX indicator.
    """
    # Create a copy of the DataFrame to avoid modifying the original data
    df = data.copy()
    df = df.sort_values('Trading Day', ascending=False)
    
    # Calculate alpha for the exponential moving average
    alpha = 1 / period

    # Calculate True Range (TR)
    df['H-L'] = df['High'] - df['Low']
    df['H-C'] = np.abs(df['High'] - df['Close'].shift(1))
    df['L-C'] = np.abs(df['Low'] - df['Close'].shift(1))
    df['TR'] = df[['H-L', 'H-C', 'L-C']].max(axis=1)
    del df['H-L'], df['H-C'], df['L-C']

    # Calculate Average True Range (ATR) using exponential moving average of TR
    df['ATR'] = df['TR'].ewm(alpha=alpha, adjust=False).mean()

    # Calculate +DM and -DM values
    df['H-pH'] = df['High'] - df['High'].shift(1)
    df['pL-L'] = df['Low'].shift(1) - df['Low']
    df['+DX'] = np.where(
        (df['H-pH'] > df['pL-L']) & (df['H-pH'] > 0),
        df['H-pH'],
        0.0
    )
    df['-DX'] = np.where(
        (df['H-pH'] < df['pL-L']) & (df['pL-L'] > 0),
        df['pL-L'],
        0.0
    )
    del df['H-pH'], df['pL-L']

    # Calculate smoothed +DM and -DM values using exponential moving average
    df['S+DM'] = df['+DX'].ewm(alpha=alpha, adjust=False).mean()
    df['S-DM'] = df['-DX'].ewm(alpha=alpha, adjust=False).mean()

    # Calculate +DMI and -DMI values as a percentage of ATR
    df['+DMI'] = (df['S+DM'] / df['ATR']) * 100
    df['-DMI'] = (df['S-DM'] / df['ATR']) * 100
    del df['S+DM'], df['S-DM']

    # Calculate Directional Movement Index (DX) and ADX using exponential moving average
    df['DX'] = (np.abs(df['+DMI'] - df['-DMI']) / (df['+DMI'] + df['-DMI'])) * 100
    df['ADX'] = df['DX'].ewm(alpha=alpha, adjust=False).mean()
    del df['DX'], df['ATR'], df['TR'], df['-DX'], df['+DX'], df['+DMI'], df['-DMI']
    
    df = df.sort_values('Trading Day', ascending=True)

    return df

# Function to calculate Bollinger Bands for a given DataFrame
def bollinger(df):
    df = df.sort_values('Trading Day', ascending=False) 

    # Calculate rolling mean and standard deviation
    window = 20  # Number of periods for rolling calculation
    df['RollingMean'] = df['Close'].rolling(window=window).mean()
    df['RollingStd'] = df['Close'].rolling(window=window).std()

    # Calculate Bollinger Bands
    df['UpperBand1'] = df['RollingMean'] + df['RollingStd']
    df['LowerBand1'] = df['RollingMean'] - df['RollingStd']
    df['UpperBand2'] = df['RollingMean'] + 2 * df['RollingStd']
    df['LowerBand2'] = df['RollingMean'] - 2 * df['RollingStd']

    df = df.sort_values('Trading Day', ascending=True)
    
    return df

#Calculate CCI
def calculate_cci(df, interval):
    df = df.sort_values('Trading Day', ascending=False)
    # Calculate the Typical Price
    df['TP'] = (df['High'] + df['Low'] + df['Close']) / 3
    
    # Calculate the moving average of the Typical Price
    df['MA_TP'] = df['TP'].rolling(window=interval).mean()
    
    # Calculate the mean deviation
    df['MD'] = df['TP'].rolling(window=interval).apply(lambda x: abs(x - x.mean()).mean())
    
    # Calculate the CCI
    df[f'CCI_{interval}'] = (df['TP'] - df['MA_TP']) / (0.015 * df['MD'])
    
    # Drop the intermediate columns used for calculations
    df.drop(['TP', 'MA_TP', 'MD'], axis=1, inplace=True)
    df = df.sort_values('Trading Day', ascending=True)   
    return df


# VPT
def calculate_vpt(df):
    df = df.sort_values('Trading Day', ascending=False)
    
    df['VPT'] = (df['Close'].pct_change() + 1).cumprod() * df['Volume']
    df['VPT_signal'] = df['VPT'].ewm(span=9, adjust=False).mean()
    
    df = df.sort_values('Trading Day', ascending=True)
    
    return df

# VWAP
def calculate_vwap(df):
    df = df.sort_values('Trading Day', ascending=False)
    
    df['cum_vol'] = df['Volume'].cumsum()
    df['cum_vol_price'] = (df['Close'] * df['Volume']).cumsum()
    df['VWAP'] = df['cum_vol_price'] / df['cum_vol']
    df.drop(['cum_vol', 'cum_vol_price'], axis=1, inplace=True)
    
    df = df.sort_values('Trading Day', ascending=True)
    
    return df

# MFI
def calculate_mfi(df, interval=14):
    df = df.sort_values('Trading Day', ascending=False)
    
    df['TP'] = (df['High'] + df['Low'] + df['Close']) / 3
    df['money_flow'] = df['TP'] * df['Volume']
    
    df['positive_money_flow'] = df['money_flow'].where(df['TP'] > df['TP'].shift(1), 0)
    df['negative_money_flow'] = df['money_flow'].where(df['TP'] < df['TP'].shift(1), 0)
    
    df['positive_money_flow'] = df['positive_money_flow'].rolling(window=interval).sum()
    df['negative_money_flow'] = df['negative_money_flow'].rolling(window=interval).sum()
    
    MFI = 100 - (100 / (1 + df['positive_money_flow'] / df['negative_money_flow']))
    df[f'MFI_{interval}'] = MFI
    
    df.drop(['TP', 'money_flow', 'positive_money_flow', 'negative_money_flow'], axis=1, inplace=True)
    
    df = df.sort_values('Trading Day', ascending=True)
    
    return df

# OBV
def calculate_obv(df):
    df = df.sort_values('Trading Day', ascending=False)
    
    df['OBV'] = (df['Volume'] * (~df['Close'].diff().le(0) * 2 - 1)).cumsum()
    
    df = df.sort_values('Trading Day', ascending=True)
    
    return df

# Pivot Points with additional levels
def calculate_pivotpoints(df):
    df['PP'] = (df['High'] + df['Low'] + df['Close']) / 3
    df['R1'] = (2 * df['PP']) - df['Low']
    df['S1'] = (2 * df['PP']) - df['High']
    df['R2'] = df['PP'] + (df['High'] - df['Low'])
    df['S2'] = df['PP'] - (df['High'] - df['Low'])
    df['R3'] = df['High'] + 2 * (df['PP'] - df['Low'])
    df['S3'] = df['Low'] - 2 * (df['High'] - df['PP'])
    return df

#calculate williams r
def calculate_williams_r(df, period=14):
    df = df.sort_values('Trading Day', ascending=False)  
    
    df['Highest High'] = df['High'].rolling(window=period).max()
    df['Lowest Low'] = df['Low'].rolling(window=period).min()
    df['%R'] = (-100) * (df['Highest High'] - df['Close']) / (df['Highest High'] - df['Lowest Low'])
    df.drop(['Highest High', 'Lowest Low'], axis=1, inplace=True)
    
    df = df.sort_values('Trading Day', ascending=True)  
    
    return df

# Chaikin Money Flow (CMF)
def calculate_cmf(df, period=20):
    df = df.sort_values('Trading Day', ascending=False)  
    
    df['Money Flow Multiplier'] = ((df['Close'] - df['Low']) - (df['High'] - df['Close'])) / (df['High'] - df['Low'])
    df['Money Flow Volume'] = df['Money Flow Multiplier'] * df['Volume']
    df['CMF'] = df['Money Flow Volume'].rolling(window=period).sum() / df['Volume'].rolling(window=period).sum()
    df.drop(['Money Flow Multiplier', 'Money Flow Volume'], axis=1, inplace=True)
    
    df = df.sort_values('Trading Day', ascending=True)
    
    return df

# Function to compile historical data and technical indicators for a single company
def compile_single(symbol):
    # Get historical data for the given symbol
    result = retrieve_api(symbol)
    historic_data = get_hist_data(result)
    
    # Calculate technical indicators
    data = bollinger(historic_data) 
    data = calculate_cci(data, 10)
    data = calculate_cci(data, 40)
    data = calculate_obv(data)
    data = calculate_vpt(data)
    data = calculate_vwap(data)
    data = calculate_mfi(data, 14)
    data = calculate_pivotpoints(data)
    data = calculate_cmf(data)
    data = calculate_williams_r(data)
    data = calc_macd(data)
    data = calc_rsi(data, 14)
    data = calc_ADX(data, 14)
    data = calc_sma(data, 5)
    data = calc_sma(data, 20)
    data = calc_sma(data, 50)
    data = calc_sma(data, 100)
    
    # Add symbol and creation timestamp columns to the DataFrame
    data['symbol'] = symbol
    data["created_on"] = datetime.now()

    # Save the DataFrame to a CSV file
    compilation_name = "historical/" + symbol.replace('.', '_') + ".csv"
    data.to_csv(compilation_name, index=False)
    
    return data

#calculate % change b/w two ends of user defined interval and qty
def calc_change(data,series,duration, changetype):
    #duartion is atuple i.e 1 month is (1, 'M'), 1 week is (1, 'W') etc.
    if duration[1]!="D":
        end = data['date_only'][0]
        start = end-np.timedelta64(duration[0], duration[1])
        value_start = data[data['date_only']>=start][series].values[-1]
        value_end = data[data['date_only']>=end][series].values[-1]
    else:
        value_end = data[data["Trading Day"]==1][series].values[0]
        value_start = data[data["Trading Day"]==1+duration[0]][series].values[0]
   
    #return (value_start,value_end)
    if changetype == "%":
        return round(((value_end - value_start) / value_start) * 100,2)
    else:
        return (value_end - value_start)

def stds(data):
    durations = [(1, "M"), (3, "M"), (6, "M"), (1, "Y")]
    end = data['date_only'][0]
    
    std_values = [
        data[data['date_only'] >= (end - np.timedelta64(duration[0], duration[1]))]['Close'].std()
        for duration in durations
    ]
    
    columns = ['std_' + str(duration[0]) + duration[1] for duration in durations]
    volatility = pd.DataFrame([std_values], columns=columns)
    
    return volatility

def calc_most_recent(data, column_list):
    return {"Latest " + col: data[col].values[0] for col in column_list}

# Function to collect key parameters and generate features
def collect_features(df):
    temp = pd.DataFrame()
       
    # Calculate percentage change for different durations and technical indicators
    changes = {}
    for j in [(1, "D"), (5, "D"), (10, "D"), (1, "M"), (3, "M"), (6, "M"), (1, "Y")]:
        for i in ['Close', 'Volume']:
            col_name = i + "_change_" + str(j[0]) + j[1].lower()
            changes[col_name] = calc_change(df, i, j, '%')
            
    for n in [(1, "D"), (5, "D"), (10, "D"),]:
        for m in ['macd', 'macd_signal','CCI_10', 'CCI_40', 'OBV', 'VPT', 'VPT_signal', 'VWAP', 'MFI_14', 'rsi', 'ADX', 'sma_5', 'sma_20', 'sma_50', 'sma_100' ]:
            col_name = m + "_change_" + str(j[0]) + j[1].lower()
            changes[col_name] = calc_change(df, m, n, "abs")
    changes = pd.DataFrame(changes, index=[0])
    
    # Calculate most recent values for specific columns
    temp_latest = calc_most_recent(df, ['Date', 'Open', 'High', 'Low', 'Close', 'Volume',
       'Dividends', 'Stock Splits', 'date_only', 'RollingMean', 'RollingStd',
       'UpperBand1', 'LowerBand1', 'UpperBand2', 'LowerBand2', 'CCI_10',
       'CCI_40', 'OBV', 'VPT', 'VPT_signal', 'VWAP', 'MFI_14', 'PP', 'R1',
       'S1', 'R2', 'S2', 'R3', 'S3', 'CMF', '%R', 'macd', 'macd_signal',
       'macde_histogram', 'rsi', 'ADX', 'sma_5', 'sma_20', 'sma_50', 'sma_100',
       'symbol', 'created_on'])
    
    temp_latest_sr = calc_most_recent(df[df['Trading Day']==2].rename(columns= {'PP': 'PP_previousday', 'R1': 'R1_previousday', 'S1': 'S1_previousday', 
                                              'R2': 'R2_previousday', 'S2': 'S2_previousday', 
                                              'R3': 'R3_previousday', 'S3': 'S3_previousday',}), 
                                   ['PP_previousday','R1_previousday','S1_previousday',
                                    'R2_previousday','S2_previousday',
                                    'R3_previousday','S3_previousday'])
    
    temp = pd.concat([temp, pd.DataFrame(temp_latest, index=[0])], axis=1)
    temp = pd.concat([temp, pd.DataFrame(temp_latest_sr, index=[0])], axis=1)
    temp = pd.concat([temp, changes], axis=1)
    temp = pd.concat([temp, stds(df)], axis=1)
    
    temp['symbol'] = df['symbol'].values[0]
        
    return temp